In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import warnings

warnings.filterwarnings("ignore")

# Tentukan device (GPU atau CPU)
# Ini akan otomatis memilih GPU Anda jika verifikasi Anda berhasil
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Menggunakan device: {device}")

# --- Load & Scale Data (Sama seperti sebelumnya) ---
df = pd.read_csv(r'C:\FRAUD-Sinta2\Data\Kaggle_Fraud\creditcard.csv')
scaler = StandardScaler()
df['Amount_scaled'] = scaler.fit_transform(df[['Amount']])
df['Time_scaled'] = scaler.fit_transform(df[['Time']])
df_scaled = df.drop(['Amount', 'Time'], axis=1)

TIME_STEPS = 10
FEATURES = df_scaled.drop('Class', axis=1).shape[1]

# --- Buat PyTorch Dataset (Pengganti DataGenerator) ---
class FraudDataset(Dataset):
    def __init__(self, df, time_steps=10):
        self.features = df.drop('Class', axis=1).values
        self.labels = df['Class'].values
        self.time_steps = time_steps

    def __len__(self):
        # Ini adalah jumlah total sekuens yang bisa kita buat
        return len(self.features) - self.time_steps + 1

    def __getitem__(self, idx):
        # Ambil satu sekuens data
        x = self.features[idx : (idx + self.time_steps)]
        # Label diambil dari data terakhir di sekuens
        y_label = self.labels[idx + self.time_steps - 1]
        
        # Konversi ke PyTorch Tensor
        x_tensor = torch.tensor(x, dtype=torch.float32)
        y_tensor = torch.tensor(y_label, dtype=torch.long)
        
        # Untuk Autoencoder, kita butuh X sebagai target juga
        return x_tensor, y_tensor 
        # (Saat training AE, kita akan abaikan y_tensor)
        # (Saat tes, kita akan gunakan y_tensor)

# --- Pisahkan Data (Sama) ---
df_normal = df_scaled[df_scaled['Class'] == 0]
df_fraud = df_scaled[df_scaled['Class'] == 1]

# Kita bagi dataframe normal untuk train/val
train_df, val_df = train_test_split(df_normal, test_size=0.2, random_state=42)

# Buat dataset HANYA UNTUK DATA NORMAL (untuk melatih AE)
train_dataset_normal = FraudDataset(train_df, time_steps=TIME_STEPS)
val_dataset_normal = FraudDataset(val_df, time_steps=TIME_STEPS)

# Buat DataLoader (Ini yang mem-batch data)
# DataLoader akan secara otomatis mengirim data ke GPU jika `pin_memory=True`
train_loader_normal = DataLoader(
    train_dataset_normal, 
    batch_size=128, 
    shuffle=True, 
)
val_loader_normal = DataLoader(
    val_dataset_normal, 
    batch_size=128, 
    shuffle=False
)

# Buat DataLoader untuk data TES CAMPURAN (semua data)
test_dataset_mixed = FraudDataset(df_scaled, time_steps=TIME_STEPS)
test_loader_mixed = DataLoader(
    test_dataset_mixed, 
    batch_size=128, 
    shuffle=False
)
print("PyTorch DataLoaders siap.")

Menggunakan device: cuda
PyTorch DataLoaders siap.


In [2]:
class LSTM_Autoencoder(nn.Module):
    def __init__(self, n_features, time_steps, embedding_dim=8):
        super(LSTM_Autoencoder, self).__init__()
        
        self.n_features = n_features
        self.time_steps = time_steps
        self.embedding_dim = embedding_dim
        
        # Encoder
        self.lstm_encoder1 = nn.LSTM(
            input_size=n_features,
            hidden_size=32,
            batch_first=True, # (Penting: [batch, seq, feature])
            num_layers=1
        )
        self.lstm_encoder2 = nn.LSTM(
            input_size=32,
            hidden_size=16,
            batch_first=True,
            num_layers=1
        )
        self.bottleneck = nn.Linear(16, embedding_dim)
        
        # Decoder
        self.decoder_linear = nn.Linear(embedding_dim, 16)
        self.lstm_decoder1 = nn.LSTM(
            input_size=16,
            hidden_size=16,
            batch_first=True,
            num_layers=1
        )
        self.lstm_decoder2 = nn.LSTM(
            input_size=16,
            hidden_size=32,
            batch_first=True,
            num_layers=1
        )
        # Output layer (TimeDistributed di Keras)
        self.output_layer = nn.Linear(32, n_features)

    def forward(self, x):
        # x shape: [batch_size, time_steps, n_features]
        
        # Encoder
        x, _ = self.lstm_encoder1(x)
        x, (hidden_n, _) = self.lstm_encoder2(x)
        # Ambil output LSTM terakhir
        x = hidden_n.squeeze(0) # Shape: [batch_size, 16]
        x = self.bottleneck(x)  # Shape: [batch_size, 8]
        
        # Decoder
        # Ulangi bottleneck vector sebanyak time_steps
        x = x.unsqueeze(1).repeat(1, self.time_steps, 1) # Shape: [batch_size, time_steps, 8]
        # (Ini adalah pengganti 'RepeatVector' di Keras)
        
        x = self.decoder_linear(x) # Shape: [batch_size, time_steps, 16]
        x, _ = self.lstm_decoder1(x)
        x, _ = self.lstm_decoder2(x)
        x = self.output_layer(x) # Shape: [batch_size, time_steps, n_features]
        
        return x

# Inisialisasi model dan kirim ke GPU
model = LSTM_Autoencoder(n_features=FEATURES, time_steps=TIME_STEPS).to(device)
loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

print("Model PyTorch LSTM-AE berhasil dibuat dan dikirim ke GPU.")
print(model)

Model PyTorch LSTM-AE berhasil dibuat dan dikirim ke GPU.
LSTM_Autoencoder(
  (lstm_encoder1): LSTM(30, 32, batch_first=True)
  (lstm_encoder2): LSTM(32, 16, batch_first=True)
  (bottleneck): Linear(in_features=16, out_features=8, bias=True)
  (decoder_linear): Linear(in_features=8, out_features=16, bias=True)
  (lstm_decoder1): LSTM(16, 16, batch_first=True)
  (lstm_decoder2): LSTM(16, 32, batch_first=True)
  (output_layer): Linear(in_features=32, out_features=30, bias=True)
)


In [3]:
N_EPOCHS = 50 # (Bisa dikurangi jika terlalu lama)

print("Memulai pelatihan...")
for epoch in range(N_EPOCHS):
    model.train() # Set model ke mode training
    total_train_loss = 0
    
    for (seq_data, _) in train_loader_normal: # (Abaikan label 'y' saat training AE)
        # Kirim data ke GPU
        seq_data = seq_data.to(device)
        
        # 1. Forward pass
        reconstructions = model(seq_data)
        loss = loss_fn(reconstructions, seq_data) # Targetnya adalah data itu sendiri
        
        # 2. Backward pass
        optimizer.zero_grad() # Nol-kan gradien
        loss.backward()       # Hitung gradien
        optimizer.step()      # Update bobot
        
        total_train_loss += loss.item()
    
    avg_train_loss = total_train_loss / len(train_loader_normal)
    
    # --- Validasi ---
    model.eval() # Set model ke mode evaluasi (penting!)
    total_val_loss = 0
    with torch.no_grad(): # (Matikan perhitungan gradien)
        for (seq_data, _) in val_loader_normal:
            seq_data = seq_data.to(device)
            reconstructions = model(seq_data)
            val_loss = loss_fn(reconstructions, seq_data)
            total_val_loss += val_loss.item()
            
    avg_val_loss = total_val_loss / len(val_loader_normal)
    
    print(f"Epoch {epoch+1}/{N_EPOCHS}, Train Loss: {avg_train_loss:.6f}, Val Loss: {avg_val_loss:.6f}")

print("Pelatihan selesai.")


Memulai pelatihan...
Epoch 1/50, Train Loss: 0.947713, Val Loss: 0.890032
Epoch 2/50, Train Loss: 0.848899, Val Loss: 0.842832
Epoch 3/50, Train Loss: 0.809157, Val Loss: 0.810647
Epoch 4/50, Train Loss: 0.784199, Val Loss: 0.792481
Epoch 5/50, Train Loss: 0.769905, Val Loss: 0.782162
Epoch 6/50, Train Loss: 0.761276, Val Loss: 0.772334
Epoch 7/50, Train Loss: 0.752361, Val Loss: 0.764139
Epoch 8/50, Train Loss: 0.746059, Val Loss: 0.760570
Epoch 9/50, Train Loss: 0.742064, Val Loss: 0.755166
Epoch 10/50, Train Loss: 0.738017, Val Loss: 0.752820
Epoch 11/50, Train Loss: 0.734765, Val Loss: 0.751232
Epoch 12/50, Train Loss: 0.731534, Val Loss: 0.746218
Epoch 13/50, Train Loss: 0.728244, Val Loss: 0.742263
Epoch 14/50, Train Loss: 0.725059, Val Loss: 0.739909
Epoch 15/50, Train Loss: 0.722408, Val Loss: 0.738916
Epoch 16/50, Train Loss: 0.720228, Val Loss: 0.737119
Epoch 17/50, Train Loss: 0.718472, Val Loss: 0.737060
Epoch 18/50, Train Loss: 0.717189, Val Loss: 0.732678
Epoch 19/50, Tra

In [4]:
print("Menentukan threshold...")
model.eval()
val_errors = []

with torch.no_grad():
    for (seq_data, _) in val_loader_normal:
        seq_data = seq_data.to(device)
        reconstructions = model(seq_data)
        
        # Hitung MSE loss per sekuens di dalam batch
        # Pindahkan kembali ke CPU untuk numpy
        errors = torch.mean((seq_data - reconstructions) ** 2, dim=(1, 2)).cpu().numpy()
        val_errors.extend(errors)

val_errors = np.array(val_errors)
threshold = np.percentile(val_errors, 99) # Ambil persentil ke-99
print(f"Threshold ditetapkan: {threshold:.6f}")

# --- Evaluasi Final pada Data Tes Campuran ---
print("Memulai evaluasi final pada data tes campuran...")
test_errors = []
all_labels = []

with torch.no_grad():
    for (seq_data, labels) in test_loader_mixed:
        seq_data = seq_data.to(device)
        reconstructions = model(seq_data)
        
        errors = torch.mean((seq_data - reconstructions) ** 2, dim=(1, 2)).cpu().numpy()
        test_errors.extend(errors)
        all_labels.extend(labels.numpy())

test_errors = np.array(test_errors)
all_labels = np.array(all_labels)

# Terapkan threshold
y_pred = [1 if e > threshold else 0 for e in test_errors]

# Evaluasi
print("\n--- Hasil LSTM-Autoencoder (PyTorch) ---")
print(classification_report(all_labels, y_pred, target_names=['Normal (0)', 'Fraud (1)']))

print("\nNotebook 2 (PyTorch) Selesai!")


Menentukan threshold...
Threshold ditetapkan: 1.587181
Memulai evaluasi final pada data tes campuran...

--- Hasil LSTM-Autoencoder (PyTorch) ---
              precision    recall  f1-score   support

  Normal (0)       1.00      0.98      0.99    284306
   Fraud (1)       0.04      0.36      0.06       492

    accuracy                           0.98    284798
   macro avg       0.52      0.67      0.53    284798
weighted avg       1.00      0.98      0.99    284798


Notebook 2 (PyTorch) Selesai!


In [ ]:
DATA_PATH = r'C:\FRAUD-Sinta2\Data\Kaggle_Fraud'
MODEL_PATH = r'C:\FRAUD-Sinta2\Output'